In [1]:
# =============================================================================
# Agente con Base de Datos: Clínica Médica
# =============================================================================
#
# Un agente que consulta una base de datos SQLite de una clínica usando
# lenguaje natural. El agente:
#
#   1. Entiende la pregunta del usuario
#   2. Decide qué tool(s) usar
#   3. Puede generar SQL arbitrario (solo SELECT — read-only por diseño)
#   4. Interpreta los resultados y responde en lenguaje natural
#
# Grafo:
#   [START] → [agent] ⇄ [tools] → [END]
#
# Tools disponibles:
#   - buscar_paciente(nombre)         → búsqueda por nombre parcial
#   - ver_historial(paciente_id)      → todas las consultas de un paciente
#   - listar_medicamentos(paciente_id)→ medicamentos activos
#   - ejecutar_sql(query)             → SQL libre, solo SELECT (text-to-SQL)
#
# La tool ejecutar_sql es la más poderosa y didáctica: el agente genera
# la consulta SQL que necesita según la pregunta del usuario.
#
# Ejecutar:
#   python day_2b_database_agent.py
#
# Instalar:
#   pip install langgraph langchain-openai langchain-core
# =============================================================================

In [2]:

import os
import sqlite3
import json
import re
from typing import Literal
from datetime import datetime, timedelta
from dotenv import load_dotenv
import random

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode

# ════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — cambia BACKEND para elegir tu proveedor
# ════════════════════════════════════════════════════════════

load_dotenv()

BACKEND = "nvidia"

CONFIGS = {
    "ollama":    {"base_url": "http://localhost:11434/v1",          "api_key": "ollama",                           "model": "qwen2.5:7b"},
    "anthropic": {"base_url": "https://api.anthropic.com/v1",      "api_key": os.getenv("ANTHROPIC_API_KEY", ""), "model": "claude-3-5-haiku-20241022"},
    "openai":    {"base_url": None,                                 "api_key": os.getenv("OPENAI_API_KEY", ""),   "model": "gpt-4o-mini"},
    "groq":      {"base_url": "https://api.groq.com/openai/v1",    "api_key": os.getenv("GROQ_API_KEY", ""),     "model": "llama-3.3-70b-versatile"},
    "nvidia":    {"base_url": "https://integrate.api.nvidia.com/v1","api_key": os.getenv("NVIDIA_API_KEY", ""),  "model": "meta/llama-3.3-70b-instruct"},
}

cfg         = CONFIGS[BACKEND]
llm_kwargs  = {"model": cfg["model"], "api_key": cfg["api_key"]}
if cfg["base_url"]:
    llm_kwargs["base_url"] = cfg["base_url"]

llm = ChatOpenAI(**llm_kwargs)
print(f"✅ Backend: {BACKEND.upper()} | Modelo: {cfg['model']}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


In [3]:
# Creamos una base de datos SQLite en memoria con datos realistas de una clínica.
# Tres tablas:
#   pacientes    — datos demográficos
#   consultas    — visitas médicas con diagnóstico
#   medicamentos — prescripciones por consulta

DB_PATH = "clinica.db"


def crear_base_de_datos(path: str = DB_PATH) -> None:
    """Crea las tablas e inserta datos de ejemplo."""
    conn = sqlite3.connect(path)
    cur  = conn.cursor()

    # ── Tablas ────────────────────────────────────────────────────────────────
    cur.executescript("""
        DROP TABLE IF EXISTS medicamentos;
        DROP TABLE IF EXISTS consultas;
        DROP TABLE IF EXISTS pacientes;

        CREATE TABLE pacientes (
            id          INTEGER PRIMARY KEY,
            nombre      TEXT NOT NULL,
            edad        INTEGER,
            sexo        TEXT,          -- 'M' | 'F'
            ciudad      TEXT,
            telefono    TEXT,
            fecha_reg   TEXT           -- fecha de registro
        );

        CREATE TABLE consultas (
            id              INTEGER PRIMARY KEY,
            paciente_id     INTEGER REFERENCES pacientes(id),
            fecha           TEXT,
            medico          TEXT,
            especialidad    TEXT,
            diagnostico     TEXT,
            notas           TEXT
        );

        CREATE TABLE medicamentos (
            id              INTEGER PRIMARY KEY,
            consulta_id     INTEGER REFERENCES consultas(id),
            nombre          TEXT,
            dosis           TEXT,
            duracion_dias   INTEGER,
            indicaciones    TEXT
        );
    """)

    # ── Datos de pacientes ────────────────────────────────────────────────────
    pacientes = [
        (1,  "Carlos Ramírez",    45, "M", "Medellín",   "3001234567", "2022-03-15"),
        (2,  "Ana Gómez",         32, "F", "Bogotá",     "3109876543", "2022-06-20"),
        (3,  "Luis Martínez",     67, "M", "Cali",       "3204567890", "2021-11-05"),
        (4,  "María Torres",      28, "F", "Medellín",   "3051112233", "2023-01-10"),
        (5,  "Jorge Herrera",     55, "M", "Barranquilla","3156667788","2022-09-18"),
        (6,  "Sofía Castillo",    41, "F", "Medellín",   "3002223344", "2023-04-02"),
        (7,  "Andrés Vargas",     73, "M", "Bogotá",     "3108889900", "2021-07-30"),
        (8,  "Laura Jiménez",     19, "F", "Pereira",    "3203334455", "2023-08-14"),
        (9,  "Ricardo Moreno",    50, "M", "Medellín",   "3055556677", "2022-12-01"),
        (10, "Valentina Cruz",    35, "F", "Manizales",  "3157778899", "2023-02-25"),
    ]
    cur.executemany(
        "INSERT INTO pacientes VALUES (?,?,?,?,?,?,?)", pacientes
    )

    # ── Datos de consultas ────────────────────────────────────────────────────
    consultas = [
        # Carlos Ramírez — hipertensión crónica
        (1,  1, "2023-01-10", "Dr. Pérez",    "Cardiología",    "Hipertensión arterial grado II",       "PA: 160/100. Ajuste de medicación."),
        (2,  1, "2023-04-15", "Dr. Pérez",    "Cardiología",    "Hipertensión arterial controlada",     "PA: 135/85. Mantener tratamiento."),
        (3,  1, "2023-09-20", "Dr. Pérez",    "Cardiología",    "Hipertensión arterial grado I",        "PA: 145/92. Refuerzo de dieta."),
        (4,  1, "2024-01-08", "Dr. Pérez",    "Cardiología",    "Hipertensión arterial controlada",     "PA: 130/80. Evolución favorable."),
        # Ana Gómez — diabetes
        (5,  2, "2023-02-20", "Dra. López",   "Endocrinología", "Diabetes mellitus tipo 2",             "Glucosa en ayunas: 180 mg/dL."),
        (6,  2, "2023-06-10", "Dra. López",   "Endocrinología", "Diabetes mellitus tipo 2 descompensada","HbA1c: 9.2%. Ajuste insulina."),
        (7,  2, "2023-11-05", "Dra. López",   "Endocrinología", "Diabetes mellitus tipo 2 compensada",  "HbA1c: 7.1%. Buen control."),
        # Luis Martínez — múltiples condiciones
        (8,  3, "2022-12-01", "Dr. Suárez",   "Medicina Interna","EPOC moderado",                      "SpO2: 92%. Espirometría programada."),
        (9,  3, "2023-03-14", "Dr. Suárez",   "Medicina Interna","EPOC moderado + infección respiratoria","Antibioticoterapia iniciada."),
        (10, 3, "2023-07-22", "Dr. Suárez",   "Medicina Interna","EPOC moderado controlado",            "Espirometría estable."),
        (11, 3, "2024-02-10", "Dr. Suárez",   "Medicina Interna","EPOC moderado",                      "Control rutinario."),
        # María Torres — consulta única
        (12, 4, "2023-05-17", "Dra. Ramos",   "Medicina General","Gripe estacional",                   "Reposo 5 días. Manejo sintomático."),
        # Jorge Herrera — dolor crónico
        (13, 5, "2023-01-25", "Dr. Varela",   "Reumatología",   "Artritis reumatoide",                 "PCR elevada. Inicio de metotrexato."),
        (14, 5, "2023-05-30", "Dr. Varela",   "Reumatología",   "Artritis reumatoide en remisión",     "Buena respuesta al tratamiento."),
        (15, 5, "2023-10-12", "Dr. Varela",   "Reumatología",   "Artritis reumatoide estable",         "Mantener esquema actual."),
        # Sofía Castillo — ansiedad
        (16, 6, "2023-06-08", "Dra. Nieto",   "Psiquiatría",    "Trastorno de ansiedad generalizada",  "Inicio de terapia cognitiva."),
        (17, 6, "2023-09-15", "Dra. Nieto",   "Psiquiatría",    "Trastorno de ansiedad en seguimiento","Mejoría parcial. Ajuste de ISRS."),
        # Andrés Vargas — cardiopatía
        (18, 7, "2022-08-20", "Dr. Pérez",    "Cardiología",    "Insuficiencia cardíaca congestiva",   "FE: 40%. Diuresis forzada."),
        (19, 7, "2023-01-15", "Dr. Pérez",    "Cardiología",    "Insuficiencia cardíaca compensada",   "FE: 45%. Estable."),
        (20, 7, "2023-06-28", "Dr. Pérez",    "Cardiología",    "Insuficiencia cardíaca compensada",   "Control ecocardiográfico."),
        (21, 7, "2024-03-05", "Dr. Pérez",    "Cardiología",    "Insuficiencia cardíaca compensada",   "Mantener tratamiento."),
        # Laura Jiménez — migraña
        (22, 8, "2023-09-02", "Dra. Ramos",   "Neurología",     "Migraña sin aura",                   "2-3 episodios/mes. Profilaxis iniciada."),
        (23, 8, "2024-01-20", "Dra. Ramos",   "Neurología",     "Migraña sin aura controlada",        "1 episodio/mes. Buena respuesta."),
        # Ricardo Moreno — hipotiroidismo
        (24, 9, "2023-02-14", "Dra. López",   "Endocrinología", "Hipotiroidismo primario",             "TSH: 8.5 mU/L. Levotiroxina iniciada."),
        (25, 9, "2023-07-10", "Dra. López",   "Endocrinología", "Hipotiroidismo en control",           "TSH: 2.1 mU/L. Dosis óptima."),
        # Valentina Cruz — lumbalgia
        (26, 10,"2023-04-18", "Dr. Ortega",   "Ortopedia",      "Lumbalgia mecánica",                  "Fisioterapia recomendada."),
        (27, 10,"2023-08-30", "Dr. Ortega",   "Ortopedia",      "Lumbalgia crónica",                   "RMN sin hernias. Manejo conservador."),
    ]
    cur.executemany(
        "INSERT INTO consultas VALUES (?,?,?,?,?,?,?)", consultas
    )

    # ── Datos de medicamentos ─────────────────────────────────────────────────
    medicamentos = [
        # Carlos Ramírez
        (1,  1,  "Losartán",        "50mg c/12h",  30,  "Tomar con o sin alimentos"),
        (2,  1,  "Amlodipino",      "5mg c/24h",   30,  "Tomar en la mañana"),
        (3,  2,  "Losartán",        "50mg c/12h",  30,  "Continuar tratamiento previo"),
        (4,  3,  "Losartán",        "100mg c/12h", 30,  "Dosis aumentada"),
        (5,  4,  "Losartán",        "100mg c/12h", 30,  "Mantener dosis"),
        (6,  4,  "Amlodipino",      "10mg c/24h",  30,  "Dosis aumentada"),
        # Ana Gómez
        (7,  5,  "Metformina",      "850mg c/12h", 30,  "Tomar con las comidas"),
        (8,  5,  "Glibenclamida",   "5mg c/24h",   30,  "Tomar antes del desayuno"),
        (9,  6,  "Metformina",      "1000mg c/12h",30,  "Dosis aumentada"),
        (10, 6,  "Insulina glargina","10 UI c/24h", 30,  "Aplicar en abdomen en la noche"),
        (11, 7,  "Metformina",      "1000mg c/12h",30,  "Mantener dosis"),
        (12, 7,  "Insulina glargina","8 UI c/24h",  30,  "Dosis reducida por buen control"),
        # Luis Martínez
        (13, 8,  "Salbutamol",      "200mcg c/6h inhalado", 30, "Solo en crisis"),
        (14, 8,  "Tiotropio",       "18mcg c/24h inhalado", 30, "Inhalador de mantenimiento"),
        (15, 9,  "Azitromicina",    "500mg c/24h",  5,   "Completar ciclo completo"),
        (16, 9,  "Prednisona",      "40mg c/24h",   7,   "Disminuir dosis gradualmente"),
        (17, 10, "Tiotropio",       "18mcg c/24h inhalado", 30, "Continuar mantenimiento"),
        (18, 11, "Tiotropio",       "18mcg c/24h inhalado", 30, "Continuar"),
        # María Torres
        (19, 12, "Acetaminofén",    "500mg c/6h",   5,   "Máximo 4g/día"),
        (20, 12, "Loratadina",      "10mg c/24h",   5,   "Para síntomas alérgicos"),
        # Jorge Herrera
        (21, 13, "Metotrexato",     "7.5mg semanal",30,  "Tomar con ácido fólico"),
        (22, 13, "Ácido fólico",    "5mg c/24h",    30,  "Excepto el día del metotrexato"),
        (23, 14, "Metotrexato",     "10mg semanal", 30,  "Dosis aumentada"),
        (24, 15, "Metotrexato",     "10mg semanal", 30,  "Mantener dosis"),
        # Sofía Castillo
        (25, 16, "Sertralina",      "50mg c/24h",   30,  "Tomar en la mañana con alimentos"),
        (26, 16, "Alprazolam",      "0.25mg c/8h",  15,  "Solo en crisis de ansiedad"),
        (27, 17, "Sertralina",      "100mg c/24h",  30,  "Dosis aumentada"),
        # Andrés Vargas
        (28, 18, "Furosemida",      "40mg c/24h",   30,  "Tomar en la mañana"),
        (29, 18, "Enalapril",       "10mg c/12h",   30,  "Monitorear presión arterial"),
        (30, 18, "Carvedilol",      "6.25mg c/12h", 30,  "Aumentar dosis gradualmente"),
        (31, 19, "Furosemida",      "20mg c/24h",   30,  "Dosis reducida"),
        (32, 20, "Furosemida",      "20mg c/24h",   30,  "Mantener"),
        (33, 21, "Furosemida",      "20mg c/24h",   30,  "Mantener"),
        (34, 21, "Carvedilol",      "12.5mg c/12h", 30,  "Dosis aumentada"),
        # Laura Jiménez
        (35, 22, "Topiramato",      "25mg c/24h",   30,  "Profilaxis de migraña"),
        (36, 22, "Ibuprofeno",      "400mg en crisis",10, "Solo en episodios agudos"),
        (37, 23, "Topiramato",      "50mg c/24h",   30,  "Dosis aumentada"),
        # Ricardo Moreno
        (38, 24, "Levotiroxina",    "50mcg c/24h",  30,  "Tomar en ayunas 30min antes del desayuno"),
        (39, 25, "Levotiroxina",    "75mcg c/24h",  30,  "Dosis ajustada"),
        # Valentina Cruz
        (40, 26, "Diclofenaco",     "50mg c/8h",    10,  "Tomar con alimentos"),
        (41, 26, "Omeprazol",       "20mg c/24h",   10,  "Protector gástrico"),
        (42, 27, "Acetaminofén",    "1g c/8h",      15,  "Primera línea analgésica"),
    ]
    cur.executemany(
        "INSERT INTO medicamentos VALUES (?,?,?,?,?,?)", medicamentos
    )

    conn.commit()
    conn.close()
    print(f"✅ Base de datos creada: {path}")
    print(f"   → {len(pacientes)} pacientes | {len(consultas)} consultas | {len(medicamentos)} medicamentos")


crear_base_de_datos()

✅ Base de datos creada: clinica.db
   → 10 pacientes | 27 consultas | 42 medicamentos


In [4]:
def get_conn() -> sqlite3.Connection:
    """Retorna una conexión a la base de datos con row_factory para dicts."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def rows_to_list(rows) -> list[dict]:
    """Convierte sqlite3.Row a lista de dicts serializables."""
    return [dict(row) for row in rows]

In [5]:
# Le mostramos al agente el esquema completo para que pueda generar SQL correcto.

DB_SCHEMA = """
Base de datos de una clínica médica. Tablas disponibles:

TABLE pacientes (
    id          INTEGER PRIMARY KEY,
    nombre      TEXT,       -- nombre completo del paciente
    edad        INTEGER,    -- edad en años
    sexo        TEXT,       -- 'M' para masculino, 'F' para femenino
    ciudad      TEXT,       -- ciudad de residencia
    telefono    TEXT,
    fecha_reg   TEXT        -- fecha de registro en formato YYYY-MM-DD
)

TABLE consultas (
    id              INTEGER PRIMARY KEY,
    paciente_id     INTEGER,    -- FK → pacientes.id
    fecha           TEXT,       -- fecha de la consulta YYYY-MM-DD
    medico          TEXT,       -- nombre del médico
    especialidad    TEXT,       -- especialidad médica
    diagnostico     TEXT,       -- diagnóstico registrado
    notas           TEXT        -- notas clínicas adicionales
)

TABLE medicamentos (
    id              INTEGER PRIMARY KEY,
    consulta_id     INTEGER,    -- FK → consultas.id
    nombre          TEXT,       -- nombre del medicamento
    dosis           TEXT,       -- dosis y frecuencia
    duracion_dias   INTEGER,    -- duración del tratamiento en días
    indicaciones    TEXT        -- instrucciones especiales
)
"""

In [7]:

@tool
def buscar_paciente(nombre: str) -> str:
    """Busca pacientes por nombre (búsqueda parcial, no sensible a mayúsculas).
    Retorna id, nombre, edad, sexo y ciudad de los pacientes encontrados.
    Úsala primero para obtener el id antes de consultar historial o medicamentos.

    Args:
        nombre: Nombre o parte del nombre del paciente, ej: 'Carlos' o 'Ramírez'.
    """
    conn = get_conn()
    rows = conn.execute(
        "SELECT id, nombre, edad, sexo, ciudad FROM pacientes "
        "WHERE nombre LIKE ? ORDER BY nombre",
        (f"%{nombre}%",)
    ).fetchall()
    conn.close()

    if not rows:
        return f"No se encontró ningún paciente con nombre '{nombre}'."
    results = rows_to_list(rows)
    return json.dumps(results, ensure_ascii=False, indent=2)


@tool
def ver_historial(paciente_id: int) -> str:
    """Retorna el historial completo de consultas de un paciente, ordenado
    por fecha descendente (más reciente primero).

    Args:
        paciente_id: ID numérico del paciente (obtenido con buscar_paciente).
    """
    conn = get_conn()

    # Verificar que el paciente existe
    paciente = conn.execute(
        "SELECT nombre FROM pacientes WHERE id = ?", (paciente_id,)
    ).fetchone()

    if not paciente:
        conn.close()
        return f"No existe ningún paciente con id={paciente_id}."

    consultas = conn.execute(
        """SELECT c.id, c.fecha, c.medico, c.especialidad,
                  c.diagnostico, c.notas
           FROM consultas c
           WHERE c.paciente_id = ?
           ORDER BY c.fecha DESC""",
        (paciente_id,)
    ).fetchall()
    conn.close()

    if not consultas:
        return f"El paciente {paciente['nombre']} no tiene consultas registradas."

    result = {
        "paciente": paciente["nombre"],
        "total_consultas": len(consultas),
        "consultas": rows_to_list(consultas),
    }
    return json.dumps(result, ensure_ascii=False, indent=2)


@tool
def listar_medicamentos(paciente_id: int) -> str:
    """Retorna todos los medicamentos prescritos a un paciente a lo largo
    de su historial, con la fecha de la consulta en que fueron recetados.

    Args:
        paciente_id: ID numérico del paciente.
    """
    conn = get_conn()

    paciente = conn.execute(
        "SELECT nombre FROM pacientes WHERE id = ?", (paciente_id,)
    ).fetchone()

    if not paciente:
        conn.close()
        return f"No existe ningún paciente con id={paciente_id}."

    meds = conn.execute(
        """SELECT m.nombre, m.dosis, m.duracion_dias, m.indicaciones,
                  c.fecha, c.diagnostico
           FROM medicamentos m
           JOIN consultas c ON m.consulta_id = c.id
           WHERE c.paciente_id = ?
           ORDER BY c.fecha DESC, m.nombre""",
        (paciente_id,)
    ).fetchall()
    conn.close()

    if not meds:
        return f"El paciente {paciente['nombre']} no tiene medicamentos registrados."

    result = {
        "paciente": paciente["nombre"],
        "total_medicamentos": len(meds),
        "medicamentos": rows_to_list(meds),
    }
    return json.dumps(result, ensure_ascii=False, indent=2)


@tool
def ejecutar_sql(query: str) -> str:
    """Ejecuta una consulta SQL de solo lectura (SELECT) sobre la base de datos.
    Úsala para preguntas analíticas o estadísticas complejas que las otras tools
    no cubren. Tablas disponibles: pacientes, consultas, medicamentos.
    Solo se permiten consultas SELECT. Máximo 50 filas por resultado.

    Args:
        query: Consulta SQL válida que empiece con SELECT.
    """
    # Guardarrail de seguridad: solo SELECT
    query_clean = query.strip().upper()
    if not query_clean.startswith("SELECT"):
        return "Error: solo se permiten consultas SELECT. No se ejecutó la query."

    # Bloquear palabras peligrosas por si acaso
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE", "ATTACH"]
    for word in forbidden:
        if word in query_clean:
            return f"Error: la query contiene la palabra '{word}' no permitida."

    try:
        conn = get_conn()
        rows = conn.execute(query).fetchmany(50)  # máximo 50 filas
        conn.close()

        if not rows:
            return "La consulta no retornó resultados."

        results = rows_to_list(rows)
        return json.dumps(results, ensure_ascii=False, indent=2)

    except sqlite3.Error as e:
        return f"Error SQL: {e}\nQuery intentada: {query}"


TOOLS = [buscar_paciente, ver_historial, listar_medicamentos, ejecutar_sql]
print(f"✅ {len(TOOLS)} tools: {[t.name for t in TOOLS]}")

✅ 4 tools: ['buscar_paciente', 'ver_historial', 'listar_medicamentos', 'ejecutar_sql']


In [8]:
SYSTEM_PROMPT = f"""Eres un asistente médico de consulta para una clínica.
Tienes acceso a la base de datos de pacientes y puedes responder preguntas
sobre historial clínico, medicamentos y estadísticas.

Flujo recomendado:
1. Si el usuario menciona un nombre, usa `buscar_paciente` para obtener el id.
2. Con el id, usa `ver_historial` o `listar_medicamentos` según la pregunta.
3. Para preguntas analíticas o estadísticas complejas, usa `ejecutar_sql`
   con una consulta SQL apropiada.

{DB_SCHEMA}

Responde siempre en español, de manera clara y profesional.
Si los datos muestran algo clínicamente relevante, menciónalo.
"""

llm_with_tools = llm.bind_tools(TOOLS)
tool_node      = ToolNode(TOOLS)


def agent_node(state: MessagesState) -> dict:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    return {"messages": [llm_with_tools.invoke(messages)]}


def router(state: MessagesState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "__end__"


builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", router)
builder.add_edge("tools", "agent")
graph = builder.compile()

print("✅ Grafo compilado.")

✅ Grafo compilado.


In [9]:
def run_agent(query: str, verbose: bool = True) -> str:
    """Ejecuta el agente con una consulta en lenguaje natural."""
    if verbose:
        print(f"\n{'═'*65}")
        print(f"👤 Consulta: {query}")
        print(f"{'─'*65}")

    final_state = None
    for step in graph.stream(
        {"messages": [HumanMessage(content=query)]},
        stream_mode="values",
    ):
        final_state = step
        if verbose:
            last = step["messages"][-1]
            cls  = last.__class__.__name__
            if hasattr(last, "tool_calls") and last.tool_calls:
                for tc in last.tool_calls:
                    args_preview = json.dumps(tc["args"], ensure_ascii=False)[:80]
                    print(f"  🔧 {tc['name']}({args_preview})")
            elif cls == "ToolMessage":
                content_preview = str(last.content)[:200]
                print(f"  📥 [{last.name}] {content_preview}{'...' if len(str(last.content))>200 else ''}")

    answer = final_state["messages"][-1].content
    if verbose:
        print(f"\n🤖 Respuesta:\n{answer}")
    return answer

In [10]:
# ── Demo 1: Búsqueda simple ───────────────────────────────────────────────
print("\n" + "█"*65)
print("DEMO 1: Búsqueda de paciente y su historial")
print("█"*65)
run_agent("¿Cuántas consultas ha tenido Carlos Ramírez y cuál es su diagnóstico más reciente?")



█████████████████████████████████████████████████████████████████
DEMO 1: Búsqueda de paciente y su historial
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: ¿Cuántas consultas ha tenido Carlos Ramírez y cuál es su diagnóstico más reciente?
─────────────────────────────────────────────────────────────────
  🔧 buscar_paciente({"nombre": "Carlos Ramírez"})
  📥 [buscar_paciente] [
  {
    "id": 1,
    "nombre": "Carlos Ramírez",
    "edad": 45,
    "sexo": "M",
    "ciudad": "Medellín"
  }
]
  🔧 ver_historial({"paciente_id": "1"})
  📥 [ver_historial] {
  "paciente": "Carlos Ramírez",
  "total_consultas": 4,
  "consultas": [
    {
      "id": 4,
      "fecha": "2024-01-08",
      "medico": "Dr. Pérez",
      "especialidad": "Cardiología",
      "di...

🤖 Respuesta:
Carlos Ramírez ha tenido 4 consultas. Su diagnóstico más reciente es hipertensión arterial controlada, con una presión arterial de

'Carlos Ramírez ha tenido 4 consultas. Su diagnóstico más reciente es hipertensión arterial controlada, con una presión arterial de 130/80 mmHg y evolución favorable.'

In [11]:
# ── Demo 2: Medicamentos ──────────────────────────────────────────────────
print("\n" + "█"*65)
print("DEMO 2: Consulta de medicamentos")
print("█"*65)
run_agent("¿Qué medicamentos le han recetado a Ana Gómez y en qué dosis?")


█████████████████████████████████████████████████████████████████
DEMO 2: Consulta de medicamentos
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: ¿Qué medicamentos le han recetado a Ana Gómez y en qué dosis?
─────────────────────────────────────────────────────────────────
  🔧 buscar_paciente({"nombre": "Ana Gómez"})
  📥 [buscar_paciente] [
  {
    "id": 2,
    "nombre": "Ana Gómez",
    "edad": 32,
    "sexo": "F",
    "ciudad": "Bogotá"
  }
]
  🔧 listar_medicamentos({"paciente_id": "2"})
  📥 [listar_medicamentos] {
  "paciente": "Ana Gómez",
  "total_medicamentos": 6,
  "medicamentos": [
    {
      "nombre": "Insulina glargina",
      "dosis": "8 UI c/24h",
      "duracion_dias": 30,
      "indicaciones": "Do...

🤖 Respuesta:
A Ana Gómez le han recetado los siguientes medicamentos:
- Insulina glargina (8 UI c/24h) desde el 5 de noviembre de 2023, con una duración de 30 días y la indica

'A Ana Gómez le han recetado los siguientes medicamentos:\n- Insulina glargina (8 UI c/24h) desde el 5 de noviembre de 2023, con una duración de 30 días y la indicación de dosis reducida por buen control.\n- Metformina (1000mg c/12h) desde el 5 de noviembre de 2023, con una duración de 30 días y la indicación de mantener dosis.\n- Insulina glargina (10 UI c/24h) desde el 10 de junio de 2023, con una duración de 30 días y la indicación de aplicar en abdomen en la noche.\n- Metformina (1000mg c/12h) desde el 10 de junio de 2023, con una duración de 30 días y la indicación de dosis aumentada.\n- Glibenclamida (5mg c/24h) desde el 20 de febrero de 2023, con una duración de 30 días y la indicación de tomar antes del desayuno.\n- Metformina (850mg c/12h) desde el 20 de febrero de 2023, con una duración de 30 días y la indicación de tomar con las comidas.'

In [12]:
# ── Demo 3: SQL analítico — el agente genera la query ────────────────────
print("\n" + "█"*65)
print("DEMO 3: Pregunta analítica (el agente genera SQL)")
print("█"*65)
run_agent("¿Cuál es el promedio de edad de los pacientes de Medellín?")


█████████████████████████████████████████████████████████████████
DEMO 3: Pregunta analítica (el agente genera SQL)
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: ¿Cuál es el promedio de edad de los pacientes de Medellín?
─────────────────────────────────────────────────────────────────
  🔧 ejecutar_sql({"query": "SELECT AVG(edad) FROM pacientes WHERE ciudad = 'Medellin'"})
  📥 [ejecutar_sql] [
  {
    "AVG(edad)": null
  }
]

🤖 Respuesta:
El promedio de edad de los pacientes de Medellín es null.


'El promedio de edad de los pacientes de Medellín es null.'

In [13]:
# ── Demo 4: Análisis cruzado ──────────────────────────────────────────────
print("\n" + "█"*65)
print("DEMO 4: Análisis cruzado de múltiples tablas")
print("█"*65)
run_agent("¿Qué pacientes están siendo tratados por el Dr. Pérez y cuál es su diagnóstico actual?")


█████████████████████████████████████████████████████████████████
DEMO 4: Análisis cruzado de múltiples tablas
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: ¿Qué pacientes están siendo tratados por el Dr. Pérez y cuál es su diagnóstico actual?
─────────────────────────────────────────────────────────────────
  🔧 ejecutar_sql({"query": "SELECT p.nombre, c.diagnostico FROM pacientes p JOIN consultas c ON p)
  📥 [ejecutar_sql] [
  {
    "nombre": "Carlos Ramírez",
    "diagnostico": "Hipertensión arterial grado II"
  },
  {
    "nombre": "Carlos Ramírez",
    "diagnostico": "Hipertensión arterial controlada"
  },
  {
    "n...

🤖 Respuesta:
Los pacientes que están siendo tratados por el Dr. Pérez son Carlos Ramírez y Andrés Vargas. El diagnóstico actual de Carlos Ramírez es hipertensión arterial grado II, y el de Andrés Vargas es insuficiencia cardíaca congestiva.


'Los pacientes que están siendo tratados por el Dr. Pérez son Carlos Ramírez y Andrés Vargas. El diagnóstico actual de Carlos Ramírez es hipertensión arterial grado II, y el de Andrés Vargas es insuficiencia cardíaca congestiva.'

In [14]:
# ── Demo 5: Estadística clínica ───────────────────────────────────────────
print("\n" + "█"*65)
print("DEMO 5: Estadística clínica")
print("█"*65)
run_agent("¿Cuál es la especialidad médica con más consultas registradas?")


█████████████████████████████████████████████████████████████████
DEMO 5: Estadística clínica
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: ¿Cuál es la especialidad médica con más consultas registradas?
─────────────────────────────────────────────────────────────────
  🔧 ejecutar_sql({"query": "SELECT especialidad FROM consultas GROUP BY especialidad ORDER BY COU)
  📥 [ejecutar_sql] [
  {
    "especialidad": "Cardiología"
  }
]

🤖 Respuesta:
La especialidad médica con más consultas registradas es la Cardiología.


'La especialidad médica con más consultas registradas es la Cardiología.'

In [16]:
# ── Demo 6: Pregunta compleja con razonamiento ────────────────────────────
print("\n" + "█"*65)
print("DEMO 6: Pregunta compleja — encadenamiento de tools")
print("█"*65)
run_agent(
    "Necesito un resumen completo de Andrés Vargas: "
    "cuántas consultas tiene, su diagnóstico actual y qué medicamentos toma."
)


█████████████████████████████████████████████████████████████████
DEMO 6: Pregunta compleja — encadenamiento de tools
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
👤 Consulta: Necesito un resumen completo de Andrés Vargas: cuántas consultas tiene, su diagnóstico actual y qué medicamentos toma.
─────────────────────────────────────────────────────────────────


  🔧 buscar_paciente({"nombre": "Andrés Vargas"})
  📥 [buscar_paciente] [
  {
    "id": 7,
    "nombre": "Andrés Vargas",
    "edad": 73,
    "sexo": "M",
    "ciudad": "Bogotá"
  }
]
  🔧 ver_historial({"paciente_id": "7"})
  📥 [ver_historial] {
  "paciente": "Andrés Vargas",
  "total_consultas": 4,
  "consultas": [
    {
      "id": 21,
      "fecha": "2024-03-05",
      "medico": "Dr. Pérez",
      "especialidad": "Cardiología",
      "di...
  🔧 listar_medicamentos({"paciente_id": "7"})
  📥 [listar_medicamentos] {
  "paciente": "Andrés Vargas",
  "total_medicamentos": 7,
  "medicamentos": [
    {
      "nombre": "Carvedilol",
      "dosis": "12.5mg c/12h",
      "duracion_dias": 30,
      "indicaciones": "Dos...

🤖 Respuesta:
Andrés Vargas tiene 4 consultas en su historial, con un diagnóstico actual de insuficiencia cardíaca compensada. Los medicamentos que toma son Carvedilol, Furosemida, Enalapril. Es importante destacar que el paciente ha tenido una evolución en su diagnóstico, des

'Andrés Vargas tiene 4 consultas en su historial, con un diagnóstico actual de insuficiencia cardíaca compensada. Los medicamentos que toma son Carvedilol, Furosemida, Enalapril. Es importante destacar que el paciente ha tenido una evolución en su diagnóstico, desde insuficiencia cardíaca congestiva a insuficiencia cardíaca compensada, lo que sugiere una mejoría en su condición de salud. Sin embargo, es fundamental continuar con el tratamiento y el monitoreo médico para asegurar el control de la enfermedad y prevenir complicaciones.'